In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install -q lightgbm mlflow dagshub

import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit

import dagshub
import mlflow
import mlflow.lightgbm

pd.set_option('display.max_columns', 50)

In [ ]:
DATA_DIR = "/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting"  
OUTPUT_DIR = "./outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_STATE = 42
N_CV_SPLITS = 3

In [ ]:
dagshub.init(repo_owner='lshek22', repo_name='walmart-recruiting-store-sales-forecasting', mlflow=True)

mlflow.set_experiment("LightGBM_v2_Training_v2")

In [ ]:
train_raw = pd.read_csv(os.path.join(DATA_DIR, "train.csv.zip"), parse_dates=["Date"])
test_raw = pd.read_csv(os.path.join(DATA_DIR, "test.csv.zip"), parse_dates=["Date"])
stores = pd.read_csv(os.path.join(DATA_DIR, "stores.csv"))
features = pd.read_csv(os.path.join(DATA_DIR, "features.csv.zip"), parse_dates=["Date"])
features["IsHoliday"] = features["IsHoliday"].astype(bool)

print("train:", train_raw.shape)
print("test:", test_raw.shape)
print("stores:", stores.shape)
print("features:", features.shape)

train_raw.head()

In [ ]:
def clean_features(features_df):
    df = features_df.copy()
    markdown_cols = [c for c in df.columns if c.startswith("MarkDown")]

    n_missing_before = int(df[markdown_cols].isna().sum().sum())
    n_negative = int((df[markdown_cols] < 0).sum().sum())

    df[markdown_cols] = df[markdown_cols].fillna(0)
    df[markdown_cols] = df[markdown_cols].clip(lower=0)

    for col in ["CPI", "Unemployment"]:
        n_missing_col = int(df[col].isna().sum())
        df[col] = df.groupby("Store")[col].transform(lambda s: s.ffill().bfill())

    stats = {
        "markdown_missing_before": n_missing_before,
        "markdown_negative_values_clipped": n_negative,
        "markdown_missing_after": int(df[markdown_cols].isna().sum().sum()),
        "cpi_unemployment_missing_after": int(df[["CPI", "Unemployment"]].isna().sum().sum()),
        "n_rows": len(df),
    }
    return df, stats


mlflow.set_experiment("LightGBM_v2_Cleaning")
with mlflow.start_run(run_name="clean_features"):
    features_clean, clean_stats = clean_features(features)

    mlflow.log_params({
        "markdown_fill_strategy": "fillna_0_clip_negative",
        "cpi_unemployment_fill_strategy": "groupby_store_ffill_bfill",
    })
    mlflow.log_metrics(clean_stats)

print(clean_stats)

In [ ]:
def merge_frame(df, stores_df, features_df):
    out = df.merge(stores_df, on="Store", how="left")
    out = out.merge(features_df.drop(columns=["IsHoliday"]), on=["Store", "Date"], how="left")
    return out


def engineer_features(train_df, test_df, stores_df, features_df):
    train_m = merge_frame(train_df, stores_df, features_df)
    test_m = merge_frame(test_df, stores_df, features_df)
    test_m["Weekly_Sales"] = np.nan

    full = pd.concat([train_m, test_m], axis=0, ignore_index=True, sort=False)
    full = full.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)

    full["Year"] = full["Date"].dt.year
    full["Month"] = full["Date"].dt.month
    full["WeekOfYear"] = full["Date"].dt.isocalendar().week.astype(int)
    full["IsHoliday"] = full["IsHoliday"].astype(bool)

    grp = full.groupby(["Store", "Dept"])["Weekly_Sales"]
    full["lag_1"] = grp.shift(1)
    full["lag_2"] = grp.shift(2)
    full["rolling_mean_4"] = (
        full.groupby(["Store", "Dept"])["Weekly_Sales"]
        .transform(lambda s: s.shift(1).rolling(4, min_periods=1).mean())
    )

    lag_cols = ["lag_1", "lag_2", "rolling_mean_4"]
    n_lag_missing_before = int(full[lag_cols].isna().sum().sum())
    for c in lag_cols:
        full[c] = full.groupby(["Store", "Dept"])[c].transform(lambda s: s.fillna(s.median()))
        full[c] = full[c].fillna(full[c].median())
    n_lag_missing_after = int(full[lag_cols].isna().sum().sum())

    stats = {
        "n_rows_total": len(full),
        "n_lag_missing_before_fill": n_lag_missing_before,
        "n_lag_missing_after_fill": n_lag_missing_after,
    }
    return full, stats


mlflow.set_experiment("LightGBM_v2_Feature_Engineering")
with mlflow.start_run(run_name="engineer_features"):
    full, fe_stats = engineer_features(train_raw, test_raw, stores, features_clean)

    mlflow.log_params({
        "lag_features": "lag_1,lag_2,rolling_mean_4",
        "lag_fillna_strategy": "groupby_median_then_global_median",
    })
    mlflow.log_metrics(fe_stats)

print(fe_stats)
full.head()

In [ ]:
CANDIDATE_FEATURES = [
    "Store", "Dept", "Type", "Size",
    "Temperature", "Fuel_Price",
    "MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5",
    "CPI", "Unemployment", "IsHoliday",
    "Year", "Month", "WeekOfYear",
    "lag_1", "lag_2", "rolling_mean_4",
]
CATEGORICAL_FEATURES = ["Store", "Dept", "Type", "IsHoliday"]

for c in CATEGORICAL_FEATURES:
    full[c] = full[c].astype("category")

train_f = full[full["Weekly_Sales"].notna()].copy()
test_f = full[full["Weekly_Sales"].isna()].copy()


def select_features(train_df, candidate_features, categorical_features, random_state=42):
    X = train_df[candidate_features]
    y = train_df["Weekly_Sales"]

    probe_model = lgb.LGBMRegressor(n_estimators=100, random_state=random_state, verbosity=-1)
    probe_model.fit(X, y, categorical_feature=categorical_features)

    importances = pd.Series(probe_model.feature_importances_, index=candidate_features)
    importances = importances.sort_values(ascending=False)
    selected = importances[importances > 0].index.tolist()
    return selected, importances


mlflow.set_experiment("LightGBM_v2_Feature_Selection")
with mlflow.start_run(run_name="select_features"):
    selected_features, importances = select_features(train_f, CANDIDATE_FEATURES, CATEGORICAL_FEATURES, RANDOM_STATE)

    mlflow.log_param("n_candidate_features", len(CANDIDATE_FEATURES))
    mlflow.log_param("n_selected_features", len(selected_features))
    mlflow.log_param("selected_features", ",".join(selected_features))
    mlflow.log_metrics({f"importance_{k}": float(v) for k, v in importances.items()})

    importance_path = os.path.join(OUTPUT_DIR, "feature_importances.csv")
    importances.to_csv(importance_path, header=["importance"])
    mlflow.log_artifact(importance_path)

selected_cat_features = [c for c in CATEGORICAL_FEATURES if c in selected_features]
print("Selected features:", selected_features)
importances

In [ ]:
def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return float(np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights))


LGBM_PARAMS = dict(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    verbosity=-1,
)

In [ ]:
train_f = train_f.sort_values("Date").reset_index(drop=True)

mlflow.set_experiment("LightGBM_v2_Training_v2")
with mlflow.start_run(run_name="cv") as parent_run:
    mlflow.log_params(LGBM_PARAMS)
    mlflow.log_param("n_cv_splits", N_CV_SPLITS)
    mlflow.log_param("cv_strategy", "TimeSeriesSplit")

    tscv = TimeSeriesSplit(n_splits=N_CV_SPLITS)
    cv_scores = []

    for fold, (tr_idx, val_idx) in enumerate(tscv.split(train_f)):
        with mlflow.start_run(run_name=f"fold_{fold}", nested=True):
            X_tr = train_f.iloc[tr_idx][selected_features]
            y_tr = train_f.iloc[tr_idx]["Weekly_Sales"]
            X_val = train_f.iloc[val_idx][selected_features]
            y_val = train_f.iloc[val_idx]["Weekly_Sales"]

            fold_model = lgb.LGBMRegressor(**LGBM_PARAMS)
            fold_model.fit(X_tr, y_tr, categorical_feature=selected_cat_features)
            preds = fold_model.predict(X_val)

            fold_wmae = wmae(y_val.values, preds, train_f.iloc[val_idx]["IsHoliday"].astype(bool).values)
            cv_scores.append(fold_wmae)

            mlflow.log_metric("wmae", fold_wmae)
            mlflow.log_metric("n_train_rows", len(X_tr))
            mlflow.log_metric("n_val_rows", len(X_val))

    mean_wmae = float(np.mean(cv_scores))
    std_wmae = float(np.std(cv_scores))
    mlflow.log_metric("cv_mean_wmae", mean_wmae)
    mlflow.log_metric("cv_std_wmae", std_wmae)

print("CV WMAE per fold:", cv_scores)
print(f"Mean CV WMAE: {mean_wmae:.2f} (+/- {std_wmae:.2f})")

In [ ]:
mlflow.set_experiment("LightGBM_v2_Training_v2")
with mlflow.start_run(run_name="final_model"):
    mlflow.log_params(LGBM_PARAMS)
    mlflow.log_param("n_train_rows", len(train_f))
    mlflow.log_param("features_used", ",".join(selected_features))

    final_model = lgb.LGBMRegressor(**LGBM_PARAMS)
    final_model.fit(
        train_f[selected_features],
        train_f["Weekly_Sales"],
        categorical_feature=selected_cat_features,
    )

    # In-sample WMAE, just as a sanity check (not a substitute for CV above)
    train_preds = final_model.predict(train_f[selected_features])
    train_wmae = wmae(train_f["Weekly_Sales"].values, train_preds, train_f["IsHoliday"].astype(bool).values)
    mlflow.log_metric("train_wmae", train_wmae)

    mlflow.lightgbm.log_model(final_model, artifact_path="model")

    # --- Predict on test set & build submission (logged as an artifact on this run) ---
    test_preds = final_model.predict(test_f[selected_features])
    test_f = test_f.copy()
    test_f["Weekly_Sales"] = test_preds

    submission = test_f.copy()
    submission["Id"] = (
        submission["Store"].astype(int).astype(str) + "_" +
        submission["Dept"].astype(int).astype(str) + "_" +
        submission["Date"].dt.strftime("%Y-%m-%d")
    )
    submission = submission[["Id", "Weekly_Sales"]]

    submission_path = os.path.join(OUTPUT_DIR, "submission.csv")
    submission.to_csv(submission_path, index=False)
    mlflow.log_artifact(submission_path)

print(f"Final model train WMAE: {train_wmae:.2f}")
print(f"Saved {len(submission)} predictions to {submission_path}")

In [ ]:
submission.head()